# Biomni Analysis 2

Run Biomni evaluations without BERTScore, LLM judge, or NLI metrics; add entity retrieval metrics to every category; and display only the overall summary table. Edit the config cell below instead of passing command line arguments.

In [6]:
from pathlib import Path

# Config parameters. These mirror the CLI arguments in biomni_analysis2.py.
EVALUATION = "biomni-base"

# None writes to evaluations/<EVALUATION>/analysis2. If the folder already has
# files, a timestamped subfolder is used unless OVERWRITE is True.
SAVE_DIR = None

# None evaluates all configured methods. Example: ["fullcontext", "ours"]
METHODS = None

# None evaluates every available prediction row.
MAX_EXAMPLES_PER_RUN = None

# None uses metrics.answer_report from config.evaluation.yaml.
ANSWER_REPORT = None

# True displays the table without writing CSV outputs.
NO_SAVE = False

# True allows writing directly into SAVE_DIR even if it already has files.
OVERWRITE = False

ROUND_DIGITS = 3

In [7]:
import importlib.util

import pandas as pd
from IPython.display import display

repo_relative_script = Path("evaluations/biomni-base/biomni_analysis2.py")
SCRIPT_PATH = repo_relative_script if repo_relative_script.exists() else Path.cwd() / "biomni_analysis2.py"
SCRIPT_PATH = SCRIPT_PATH.resolve()

spec = importlib.util.spec_from_file_location("biomni_analysis2_module", SCRIPT_PATH)
biomni = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(biomni)

biomni.load_dotenv(biomni.REPO_ROOT / ".env")
SCRIPT_PATH

PosixPath('/home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/biomni_analysis2.py')

In [8]:
evaluation_dir = biomni.resolve_repo_path(Path("evaluations") / EVALUATION)
config_path = evaluation_dir / "config.evaluation.yaml"
if not config_path.exists():
    raise FileNotFoundError(f"Evaluation config not found: {config_path}")

evaluation_config = biomni.load_config(str(config_path))
settings = evaluation_config.get("evaluation", {})
metrics_config = settings.get("metrics", {})
metrics_by_qtype = biomni.configured_metrics(
    metrics_config.get("enabled_by_qtype", {}),
)

prediction_dirs = dict(settings.get("prediction_dirs", {}))
configured_prediction_files = dict(settings.get("prediction_files", {}))
available_methods = set(prediction_dirs) | set(configured_prediction_files)
selected_methods = set(METHODS) if METHODS is not None else None
if selected_methods is not None:
    unknown_methods = selected_methods - available_methods
    if unknown_methods:
        raise ValueError(f"Unknown methods: {sorted(unknown_methods)}")

prediction_files = biomni.resolve_prediction_files(
    prediction_dirs,
    configured_prediction_files,
    settings.get("prediction_filename", "RESULTS.jsonl"),
    selected_methods,
)

answer_report = ANSWER_REPORT or metrics_config.get(
    "answer_report",
    settings.get("answer_report", "original"),
)
input_augmentations = biomni.build_input_augmentation_map(
    selected_methods,
    settings,
    answer_report,
)

gt_bundle = biomni.load_ground_truth_bundle(
    EVALUATION,
    settings.get("source_config", "config.fullcontext.yaml"),
)

print(f"Config: {config_path}")
print(f"Ground truth: {gt_bundle['ground_truth_path']}")
print(f"Ground-truth examples: {len(gt_bundle['records'])}")
print(f"Enabled qtype metrics: {', '.join(sorted(metrics_by_qtype))}")
print("Removed metrics: bert_score, llm_answer_quality, nli_entailment")
print("Forced metric on every qtype: entity_retrieval")

Config: /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/config.evaluation.yaml
Ground truth: /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/ground_truth/ground_truth_data.jsonl
Ground-truth examples: 106
Enabled qtype metrics: bool, entity, numeric
Removed metrics: bert_score, llm_answer_quality, nli_entailment
Forced metric on every qtype: entity_retrieval


In [9]:
prediction_runs = biomni.load_prediction_runs(
    prediction_files,
    input_augmentations,
    MAX_EXAMPLES_PER_RUN,
)

evaluation_rows = {}
for run_name, predictions in prediction_runs.items():
    run_rows = biomni.evaluate_run(run_name, predictions, metrics_by_qtype, gt_bundle)
    for category, rows in run_rows.items():
        evaluation_rows.setdefault(category, []).extend(rows)

categories_to_write = sorted(set(metrics_by_qtype) | set(evaluation_rows))
results_by_category = {}
summaries_by_category = {}
for category in categories_to_write:
    results_df = biomni.sort_results_df(pd.DataFrame(evaluation_rows.get(category, [])))
    summary_df = biomni.build_run_summary(results_df)
    results_by_category[category] = results_df
    summaries_by_category[category] = summary_df
    print(f"{category}: {len(results_df)} evaluated rows")

overall_summary_df = biomni.build_overall_summary(results_by_category)
overall_table = biomni.format_overall_table(overall_summary_df, ROUND_DIGITS)

display(overall_table)

ours: 100%|██████████| 106/106 [00:12<00:00,  8.24it/s]

bool: 273 evaluated rows
entity: 322 evaluated rows
numeric: 147 evaluated rows


,Method,N,Matched GT,Answer Token F1,GT Entity Coverage,Entity Recall Final,Entity Precision Final,Entity F1 Final,Entity Recall Total,Entity Precision Total,Entity F1 Total
0,FCB,106,106,37.884,9.113,0.000,0.000,NaN,0.000,0.000,NaN
1,GWB,106,106,27.993,11.987,33.600,4.809,15.318,36.725,5.194,14.918
2,VSB,106,106,25.560,14.490,5.636,2.245,16.031,15.726,2.640,13.202
3,GRASP,106,106,12.514,3.686,0.962,4.688,58.333,0.962,4.688,58.333
4,HippoRAG,106,106,9.300,21.445,39.140,0.985,3.368,66.576,0.711,1.757
5,HyperGRAG,106,106,35.557,21.442,0.000,NaN,NaN,0.000,NaN,NaN
6,Ours,106,106,35.665,30.884,46.923,4.712,12.449,55.684,1.855,5.067


In [10]:
output_dir = None
if not NO_SAVE:
    default_save_dir = evaluation_dir / "analysis2"
    requested_save_dir = biomni.resolve_repo_path(SAVE_DIR) if SAVE_DIR else default_save_dir
    output_dir = biomni.prepare_output_dir(requested_save_dir, overwrite=OVERWRITE)
    biomni.write_outputs(output_dir, results_by_category, summaries_by_category, overall_table)
    print(f"Wrote outputs under: {output_dir}")
else:
    print("NO_SAVE is True, so no files were written.")

Wrote outputs under: /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/analysis2/run_20260507_174341
